# 00 – Datencheck: Feldinventur der Lauf-FITs

**Frage:** Welche Felder liefern die FIT-Dateien meiner Läufe tatsächlich, und wie vollständig sind sie?

**Warum:** Ob eine Messgröße in die Analysen eingeht (z. B. Atemfrequenz in M4), hängt davon ab, ob die Venu 4 sie aufzeichnet und ob die Personal API sie mitliefert. Außerdem prüfen wir die Einheiten, bevor die Transformation (M2) sie umrechnet.

**Methode:** Alle FIT-Dateien mit `sport = running` dekodieren und je Nachrichtentyp (`record`, `lap`, `session`) pro Feld zählen, wie viele Werte befüllt sind.

In [ ]:
import polars as pl

from myfitness.config import RAW_FIT
from myfitness.transform.fit import decode_fit_zip, field_inventory, sport_of, to_frame

pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_width_chars(140)

## Inventur über alle Läufe

Zwei Kennzahlen pro Feld:

- `filled_share`: Anteil befüllter Werte an allen Nachrichten dieses Typs, auch aus Dateien, in denen das Feld ganz fehlt
- `files_share`: Anteil der Läufe, in denen das Feld mindestens einen Wert hat

Das Dekodieren dauert etwas, weil das SDK in reinem Python arbeitet.

In [ ]:
fit_paths = sorted(RAW_FIT.glob("*.zip"))
inventory = field_inventory(fit_paths, sport="running")
print(f"{len(fit_paths)} FIT-Dateien insgesamt")

### Record-Felder (ca. 1 Messpunkt pro Sekunde)

Zahlen statt Namen (z. B. `107`, `135`) sind Felder, die im öffentlichen FIT-Profil nicht beschrieben sind. Sie bleiben in der Analyse außen vor.

In [ ]:
inventory.filter(pl.col("message") == "record_mesgs")

### Gezielt geprüfte Felder

Atemfrequenz, Laufdynamik, Temperatur und VO2max-bezogene Felder aus `PLAN.md` 1.7. Fehlt ein Feld in der Tabelle, kommt es in keinem Lauf vor.

In [ ]:
fields_of_interest = [
    "enhanced_respiration_rate",
    "respiration_rate",
    "step_length",
    "vertical_oscillation",
    "vertical_ratio",
    "stance_time",
    "stance_time_balance",
    "stance_time_percent",
    "temperature",
    "heart_rate",
    "cadence",
    "fractional_cadence",
    "position_lat",
]
(
    inventory
    .filter(pl.col("message") == "record_mesgs", pl.col("field").is_in(fields_of_interest))
    .select("field", "filled_share", "files_share")
)

### Session-Felder (1 Zeile pro Lauf)

Hier interessieren vor allem Felder zu Temperatur, Atemfrequenz, Training Effect und VO2max.

In [ ]:
session_pattern = "(?i)temperature|respiration|vo2|training|recovery|effect"
(
    inventory
    .filter(pl.col("message") == "session_mesgs", pl.col("field").str.contains(session_pattern))
    .select("field", "filled_share", "files_share")
)

In [ ]:
# All session fields, for reference
inventory.filter(pl.col("message") == "session_mesgs").select("field", "files_share")

## Einheiten-Check an einem Lauf

Zu prüfen (siehe `CLAUDE.md`, „FIT-Fallstricke“):

1. **Koordinaten:** Liegen `position_lat`/`position_long` als Semicircles vor (Beträge um 6 · 10⁸) oder schon in Grad (in Deutschland um 50 bzw. 7–13)?
2. **Kadenz:** Ist `cadence` pro Bein angegeben (Median um 80–90) oder schon in Schritten pro Minute (um 160–180)?
3. **Laufdynamik:** Größenordnungen von `step_length`, `vertical_oscillation` (mm), `stance_time` (ms), `vertical_ratio` (%).

In [ ]:
newest_run_path = next(p for p in reversed(fit_paths) if sport_of(decode_fit_zip(p)) == "running")
records = to_frame(decode_fit_zip(newest_run_path)["record_mesgs"])

unit_columns = [
    "position_lat",
    "position_long",
    "cadence",
    "fractional_cadence",
    "enhanced_speed",
    "heart_rate",
    "step_length",
    "vertical_oscillation",
    "vertical_ratio",
    "stance_time",
    "enhanced_respiration_rate",
]
present_columns = [c for c in unit_columns if c in records.columns]
records.select(present_columns).describe()

In [ ]:
records.select("timestamp").head(3)

## Abgleich der Transformation mit Garmin Connect (M2)

Die Tabelle `activities` übernimmt Distanz, Dauer und Ø-HF aus Garmins Zusammenfassung, also genau die Werte, die Connect anzeigt. Die Frage ist daher: **Passen die selbst umgerechneten Records dazu?** Weichen sie ab, steckt ein Fehler in der Transformation, etwa bei Einheiten oder Zeitstempeln.

Pro Lauf vergleichen wir:

- Distanz: letzter Record-Wert gegen `distance_m`
- Ø-HF: Mittelwert der Records gegen `avg_hr`. Kleine Abweichungen sind normal, weil die Records Pausen enthalten.
- Zeitspanne: erster bis letzter Record gegen `elapsed_s`
- Kadenz: Median der Records (Erwartung für Läufe etwa 150–190 spm)

Zielwert: Abweichungen höchstens ±1 %.

In [ ]:
from myfitness.config import PROCESSED

activities = pl.read_parquet(PROCESSED / "activities.parquet")
records = pl.read_parquet(PROCESSED / "records.parquet")

per_run_from_records = records.group_by("activity_id").agg(
    record_distance_m=pl.col("distance_m").max(),
    record_avg_hr=pl.col("heart_rate").mean(),
    record_span_s=pl.col("elapsed_s").max(),
    record_median_cadence_spm=pl.col("cadence_spm").median(),
)

runs = activities.filter(pl.col("sport") == "running")
comparison = runs.join(per_run_from_records, on="activity_id").select(
    "activity_id",
    "start_local",
    distance_dev_pct=(pl.col("record_distance_m") / pl.col("distance_m") - 1) * 100,
    hr_dev_pct=(pl.col("record_avg_hr") / pl.col("avg_hr") - 1) * 100,
    span_dev_pct=(pl.col("record_span_s") / pl.col("elapsed_s") - 1) * 100,
    median_cadence_spm=pl.col("record_median_cadence_spm"),
)
comparison.describe()

Läufe mit einer Abweichung über 1 % (leer = alles im Rahmen):

In [ ]:
deviation_columns = ["distance_dev_pct", "hr_dev_pct", "span_dev_pct"]
comparison.filter(pl.any_horizontal(pl.col(deviation_columns).abs() > 1))

### Wetter

Für wie viele Aktivitäten mit GPS liegt eine Temperatur vor? Das Open-Meteo-Archiv hinkt einige Tage hinterher, deshalb fehlen die jüngsten Aktivitäten.

In [ ]:
activities.select(
    with_gps=pl.col("start_lat").is_not_null().sum(),
    with_temperature=pl.col("temp_c").is_not_null().sum(),
    temp_min=pl.col("temp_c").min(),
    temp_max=pl.col("temp_c").max(),
)

## Ergebnis

Die Auswertung steht in `PLAN.md`, Abschnitt 10.